In [1]:
import pandas as pd
import numpy as np
from unidecode import unidecode
from pathlib import Path
import re
import hashlib

pd.set_option('future.no_silent_downcasting', True) # Evita avisos de downcasting

# Buscar a pasta onde está o parquet
pasta = Path("../data/01_bronze/Medicamentos")
arquivo_parquet = next(pasta.glob("*.parquet"))
df = pd.read_parquet(arquivo_parquet)

In [2]:
# Normalizar as linhas
def normalize_rows(series):
    series = series.astype(str).str.strip()  # limpa espaços
    series = series.replace(["nan", "None", "NaT", "NULL"], np.nan)  # textos inválidos
    series = series.apply(lambda x: unidecode(x) if pd.notna(x) else x)
    series = series.str.upper()
    return series

for col in df.select_dtypes(include=['object']).columns:
    df[col] = normalize_rows(df[col])

In [3]:
# Converter datas no formato YYYY-MM-DD
def normalize_date_column(series):
    # Converte strings ou números para datetime
    series = pd.to_datetime(series.astype(str).str.strip(), errors="coerce", format="%Y%m%d")
    # Mantém apenas a parte da data (sem hora)
    series = series.dt.floor("d")  # arredonda para o dia
    return series

colunas_data = ["INICIO_ADMINISTRACAO", "FIM_ADMINISTRACAO"]

for col in colunas_data:
    if col in df.columns:
        df[col] = normalize_date_column(df[col])

In [4]:
def mapear_coluna(df, coluna, mapa, nome_valor=None, tipo_int64=True):
    if coluna not in df.columns:
        return df

    nome_valor = nome_valor or f"{coluna}_VALOR"
    df[nome_valor] = df[coluna]

    # Substitui pelos códigos
    df[coluna] = df[coluna].replace(mapa)
    
    if tipo_int64:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0).astype("Int64")

    return df

In [5]:
def extrair_numero(df, coluna, nome_valor=None, nome_unidade=None, tipo_valor="float"):
    if coluna not in df.columns:
        return df

    nome_valor = nome_valor or f"{coluna}_VALOR"
    nome_unidade = nome_unidade or f"{coluna}_UNIDADE"

    def extrair(valor):
        if pd.isna(valor):
            return (np.nan, np.nan)
        texto = str(valor).strip().upper()
        match = re.match(r"([\d.,]+)\s*([A-ZÇÃÉÍ]*)?", texto)
        if not match:
            return (np.nan, np.nan)
        numero_str = match.group(1)

        # Remove pontos de milhar e substitui vírgula por ponto
        numero_str = numero_str.replace(".", "").replace(",", ".")
        try:
            numero = float(numero_str) if tipo_valor == "float" else int(float(numero_str))
        except:
            numero = np.nan

        unidade = match.group(2) if match.group(2) else ""
        unidade = ("ANO(S)" if "ANO" in unidade else 
                  "MES(ES)" if "MES" in unidade else 
                  "DIA(S)" if "DIA" in unidade else 
                  "HORA(S)" if "HORA" in unidade else 
                  "MINUTO(S)" if "MINUTO" in unidade else 
                  "MILLIGRAM (MG)" if "MG" in unidade else 
                  "INTERNATIONAL UNIT (IU)" if "IU" in unidade else 
                  "MILLILITRE (ML)" if "ML" in unidade else 
                  "GRAM (G)" if "G" in unidade else 
                  "MICROGRAM (UG)" if "UG" in unidade else 
                  "NAN")
        return (numero, unidade)

    df[[nome_valor, nome_unidade]] = df[coluna].apply(lambda x: pd.Series(extrair(x)))

    if tipo_valor == "float":
        df[nome_valor] = df[nome_valor].astype("float64")
    else:
        df[nome_valor] = df[nome_valor].fillna(0).astype("Int64")

    return df

df = extrair_numero(df, "DOSE", tipo_valor="float")
df = extrair_numero(df, "DURACAO", tipo_valor="int")

In [6]:
def inferir_tipos(df):
    tipos = {}
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            tipos[col] = "Int64" if pd.api.types.is_integer_dtype(df[col]) else "float64"
        elif pd.api.types.is_object_dtype(df[col]):
            sample = df[col].dropna().head(20).astype(str)
            if all(pd.to_datetime(sample, format="%Y%m%d", errors="coerce").notna()):
                tipos[col] = "datetime64[ns]"
            else:
                tipos[col] = "string"
        else:
            tipos[col] = str(df[col].dtype)
    return tipos

In [10]:
df["PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO"].value_counts(dropna=False)

PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO
NaN                                                                               534294
ERRO DE MEDICACAO                                                                   7969
USO "OFF-LABEL" USO SEM REGISTRO                                                    3407
LOTES TESTADOS E DENTRO DAS ESPECIFICACOES                                          1286
SUPERDOSE                                                                            915
                                                                                   ...  
FALSIFICACAO_X000D_|USO "OFF-LABEL" USO SEM REGISTRO                                   1
SUPERDOSE_X000D_|ERRO DE MEDICACAO_X000D_|                                             1
USO INCORRETO_X000D_|LOTES TESTADOS E DENTRO DAS ESPECIFICACOES                        1
USO "OFF-LABEL" USO SEM REGISTRO_X000D_|ERRO DE MEDICACAO_X000D_|USO INCORRETO         1
SUPERDOSE_X000D_|USO "OFF-LABEL" USO SEM REGISTRO_X000D_|USO INCO

In [11]:
# COMPONENTE_SUSPEITO
componente_suspeito_map = {
    "PRINCIPIO ATIVO": 1, "EXCIPIENTE, NAO CLASSIFICADO": 2, "SOLVENTE": 3, "CORANTE": 4,
    "CONSERVANTE": 5, "AGENTE FLAVORIZADOR": 6, "EXCESSO PERCENTUAL": 7, "ANTIOXIDANTE": 8,
    "ESTABILIZANTE": 9, "VALOR AUSENTE": 0
}

# PROBLEMAS_ADD_REL_MEDICAMENTO
problemas_add_rel_medicamento_map = {
    "EXPOSICAO OCUPACIONAL": 1, "ERRO DE MEDICACAO": 2,
    "LOTES TESTADOS E DENTRO DAS ESPECIFICACOES": 3, 'USO "OFF-LABEL" USO SEM REGISTRO': 4,
    "USO INCORRETO": 5, "SUPERDOSE": 6, "ERRO DE MEDICACAO, USO INCORRETO": 7, "ABUSO": 8, 
    "FALSIFICACAO, ERRO DE MEDICACAO, USO INCORRETO, ABUSO": 9, "OUTROS": 10, "NAO INFORMADO": 0
}

df = mapear_coluna(df, "COMPONENTE_SUSPEITO", componente_suspeito_map)
df = mapear_coluna(df, "PROBLEMAS_ADD_REL_MEDICAMENTO", problemas_add_rel_medicamento_map)

In [12]:
# Caminho do arquivo Gold
caminho_gold = "../data/03_gold/dim_medicamentos/dim_medicamentos.parquet"
df.to_parquet(caminho_gold, index=False)